# Chapter 5.1: SGLang Codebase Tour

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the SGLang repository structure** and how it is organized into major subsystems
2. **Identify key modules**: `srt` (SGLang Runtime) and `lang` (Frontend DSL)
3. **Trace entry points** from `python -m sglang.launch_server` to actual server startup
4. **Understand configuration classes** like `ServerArgs` and `ModelConfig`
5. **Map the dependency graph** between SGLang components
6. **Compare SGLang vs vLLM** design philosophies at the architectural level

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.1_codebase_tour.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.1_codebase_tour.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. SGLang at a Glance

SGLang (pronounced "S-G-Lang") is a structured generation language framework for large language models. It combines:

- **A frontend DSL** (`sglang.lang`) for writing LLM programs with primitives like `gen()`, `select()`, `fork()`
- **A high-performance runtime** (`sglang.srt`) that serves LLM inference with innovations like RadixAttention

The project originated from the research paper *"SGLang: Efficient Execution of Structured Language Model Programs"* from UC Berkeley.

### Why SGLang Matters

| Feature | SGLang | vLLM |
|---------|--------|------|
| KV-Cache Reuse | RadixAttention (automatic prefix caching via radix tree) | PagedAttention (manual prefix caching) |
| Frontend DSL | First-class structured generation language | OpenAI-compatible API only |
| Constrained Decoding | Native JSON/Regex/Grammar with jump-forward | Via Outlines/XGrammar integration |
| Attention Backend | FlashInfer-first | FlashAttention/FlashInfer |
| Scheduling | Cache-aware scheduling | Priority-based with preemption |

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesfrom matplotlib.patches import FancyBboxPatch, FancyArrowPatchfig, ax = plt.subplots(1, 1, figsize=(14, 8))ax.set_xlim(0, 14)ax.set_ylim(0, 10)ax.set_aspect('equal')ax.axis('off')fig.patch.set_facecolor('white')# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'# Component boxescomponents = [    (1, 7.5, 2.5, 1.2, "HTTP Server\n(FastAPI)", BLUE),    (4.5, 7.5, 2.5, 1.2, "Router /\nTokenizerManager", GREEN),    (8, 7.5, 2.5, 1.2, "DetokenizerManager", GREEN),    (4.5, 5, 2.5, 1.2, "Scheduler", ORANGE),    (4.5, 2.5, 2.5, 1.2, "TpModelWorker", RED),    (4.5, 0.5, 2.5, 1.2, "ModelRunner\n(GPU Forward)", PURPLE),    (9, 5, 2.5, 1.2, "RadixCache", ORANGE),    (9, 2.5, 2.5, 1.2, "MemoryPool\n(KV Cache)", RED),    (0.5, 3.5, 2.2, 1.2, "FlashInfer\nBackend", PURPLE),]for x, y, w, h, label, color in components:    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",                         facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)    ax.add_patch(box)    ax.text(x + w/2, y + h/2, label, ha='center', va='center',            fontsize=9, fontweight='bold', color='white')# Arrowsarrow_style = dict(arrowstyle='->', color=DARK_GRAY, lw=2, connectionstyle='arc3,rad=0')arrows = [    ((2.25, 8.1), (4.5, 8.1)),   # HTTP -> Router    ((7.0, 8.1), (8.0, 8.1)),    # Router -> Detokenizer    ((5.75, 7.5), (5.75, 6.2)),  # Router -> Scheduler    ((5.75, 5.0), (5.75, 3.7)),  # Scheduler -> TpModelWorker    ((5.75, 2.5), (5.75, 1.7)),  # TpModelWorker -> ModelRunner    ((7.0, 5.6), (9.0, 5.6)),   # Scheduler -> RadixCache    ((9.5, 5.0), (9.5, 3.7)),   # RadixCache -> MemoryPool    ((4.5, 1.1), (2.7, 2.5)),   # ModelRunner -> FlashInfer (left)    ((4.5, 3.1), (2.7, 3.8)),   # TpModelWorker -> FlashInfer]for (x1, y1), (x2, y2) in arrows:    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),                arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=1.8))# Titleax.text(7, 9.5, 'SGLang Architecture: Component Diagram',        ha='center', va='center', fontsize=14, fontweight='bold', color=DARK_GRAY)# Legendlegend_items = [    ("Entry Point", BLUE), ("Manager Process", GREEN),    ("Scheduling & Cache", ORANGE), ("GPU Worker", RED),    ("GPU Execution", PURPLE)]for i, (label, color) in enumerate(legend_items):    ax.add_patch(plt.Rectangle((0.5 + i*2.5, -0.3), 0.3, 0.3, facecolor=color, alpha=0.85))    ax.text(0.9 + i*2.5, -0.15, label, fontsize=8, va='center', color=DARK_GRAY)plt.tight_layout()plt.savefig("sglang_architecture.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

## 2. Repository Structure Overview

The SGLang repository is organized as follows:

```
sglang/
├── python/
│   └── sglang/
│       ├── __init__.py          # Package init, version
│       ├── api.py               # High-level Python API
│       ├── launch_server.py     # Server entry point
│       ├── version.py           # Version string
│       ├── lang/                # Frontend DSL
│       │   ├── __init__.py
│       │   ├── backend/         # Execution backends
│       │   ├── chat_template.py
│       │   ├── compiler.py
│       │   ├── interpreter.py
│       │   └── ir.py            # Intermediate representation
│       └── srt/                 # SGLang Runtime
│           ├── __init__.py
│           ├── configs/         # Model and server configs
│           ├── constrained/     # Constrained decoding
│           ├── layers/          # Model layers
│           ├── managers/        # Tokenizer, Detokenizer, Scheduler
│           ├── mem_cache/       # Memory/KV-cache management
│           ├── model_executor/  # Model execution
│           ├── models/          # Model implementations
│           ├── openai_api/      # OpenAI-compatible API
│           ├── sampling/        # Sampling strategies
│           └── server.py        # FastAPI server
├── test/                        # Test suite
├── benchmark/                   # Benchmarking scripts
├── docs/                        # Documentation
└── setup.py / pyproject.toml    # Build configuration
```

Let's explore each major component.

In [ ]:
# Let's simulate exploring the SGLang directory structure
# In a real environment, you'd clone the repo first:
# !git clone https://github.com/sgl-project/sglang.git

import os

def show_tree(path, prefix="", max_depth=3, current_depth=0):
    """Display directory tree structure."""
    if current_depth >= max_depth:
        return
    
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    
    # Filter out __pycache__, .git, etc.
    entries = [e for e in entries if not e.startswith('.') and e != '__pycache__']
    
    for i, entry in enumerate(entries):
        full_path = os.path.join(path, entry)
        is_last = (i == len(entries) - 1)
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{entry}")
        
        if os.path.isdir(full_path):
            extension = "    " if is_last else "│   "
            show_tree(full_path, prefix + extension, max_depth, current_depth + 1)

# If sglang is installed, show its package structure
try:
    import sglang
    sglang_path = os.path.dirname(sglang.__file__)
    print(f"SGLang package location: {sglang_path}\n")
    show_tree(sglang_path, max_depth=2)
except ImportError:
    print("SGLang not installed. Showing expected structure...")
    expected = {
        "sglang/": {
            "__init__.py": None,
            "api.py": None,
            "launch_server.py": None,
            "lang/": {"backend/": {}, "ir.py": None, "interpreter.py": None},
            "srt/": {"configs/": {}, "managers/": {}, "models/": {}, "server.py": None}
        }
    }
    print("Expected structure:")
    for k, v in expected.items():
        print(f"  {k}")

## 3. The Two Pillars: `srt` and `lang`

### 3.1 `sglang.srt` — The SGLang Runtime

The runtime (`srt`) is the **inference engine**. It handles:

- **Server**: FastAPI-based HTTP server with OpenAI-compatible endpoints
- **Scheduler**: Request scheduling with cache-aware policies
- **Memory Management**: RadixAttention-based KV-cache with radix tree
- **Model Execution**: Forward pass execution with tensor parallelism
- **Tokenization**: Async tokenization/detokenization management
- **Sampling**: Top-k, top-p, temperature, repetition penalty, etc.
- **Constrained Decoding**: JSON, regex, grammar-guided generation

```
srt/
├── server.py              # FastAPI HTTP server
├── server_args.py         # ServerArgs configuration
├── managers/
│   ├── tokenizer_manager.py
│   ├── detokenizer_manager.py
│   ├── scheduler.py       # Core scheduler logic
│   ├── tp_worker.py       # Tensor parallel worker
│   └── io_struct.py       # I/O data structures
├── mem_cache/
│   ├── radix_cache.py     # RadixAttention implementation
│   ├── memory_pool.py     # GPU memory pool
│   └── chunk_cache.py     # Chunked cache management
├── model_executor/
│   ├── model_runner.py    # Forward pass execution
│   └── forward_batch_info.py  # Batch metadata
├── models/                # Model implementations
│   ├── llama.py
│   ├── qwen2.py
│   ├── mixtral.py
│   └── ...               # Many more models
├── layers/
│   ├── attention/         # Attention backends
│   ├── logits_processor.py
│   ├── sampler.py
│   └── quantization/     # Quantization support
└── sampling/
    ├── sampling_batch_info.py
    └── sampling_params.py
```

### 3.2 `sglang.lang` — The Frontend DSL

The DSL (`lang`) provides a **programming language** for LLM interactions:

```
lang/
├── __init__.py            # Exports: gen, select, function, etc.
├── ir.py                  # Intermediate representation nodes
├── interpreter.py         # Interprets SGLang programs
├── compiler.py            # Compiles for optimized execution
├── chat_template.py       # Chat template management
└── backend/
    ├── base_backend.py    # Abstract backend interface
    ├── runtime_endpoint.py # SRT backend
    ├── openai.py          # OpenAI API backend
    └── anthropic.py       # Anthropic API backend
```

The DSL allows writing **structured LLM programs**:

```python
@sgl.function
def multi_step_reasoning(s, question):
    s += sgl.system("You are a helpful assistant.")
    s += sgl.user(question)
    s += sgl.assistant(sgl.gen("analysis", max_tokens=200))
    s += sgl.user("Now summarize in one sentence.")
    s += sgl.assistant(sgl.gen("summary", max_tokens=50))
```

In [ ]:
# Simulated module dependency analysis
# This shows the import relationships between SGLang modules

module_dependencies = {
    "sglang.launch_server": ["sglang.srt.server", "sglang.srt.server_args"],
    "sglang.api": ["sglang.lang", "sglang.srt.server"],
    
    # SRT dependencies
    "sglang.srt.server": [
        "sglang.srt.managers.tokenizer_manager",
        "sglang.srt.managers.scheduler",
        "sglang.srt.managers.detokenizer_manager",
        "sglang.srt.openai_api",
        "sglang.srt.server_args",
    ],
    "sglang.srt.managers.scheduler": [
        "sglang.srt.mem_cache.radix_cache",
        "sglang.srt.managers.tp_worker",
        "sglang.srt.managers.io_struct",
        "sglang.srt.sampling",
    ],
    "sglang.srt.managers.tp_worker": [
        "sglang.srt.model_executor.model_runner",
    ],
    "sglang.srt.model_executor.model_runner": [
        "sglang.srt.models",
        "sglang.srt.layers.attention",
        "sglang.srt.mem_cache.memory_pool",
    ],
    
    # Lang dependencies
    "sglang.lang.interpreter": [
        "sglang.lang.ir",
        "sglang.lang.backend",
    ],
    "sglang.lang.backend.runtime_endpoint": [
        "sglang.srt.server",  # connects to SRT
    ],
}

print("SGLang Module Dependency Graph")
print("=" * 50)
for module, deps in module_dependencies.items():
    print(f"\n{module}")
    for dep in deps:
        print(f"  └── imports {dep}")

## 4. Entry Points

### 4.1 Server Launch: `python -m sglang.launch_server`

The primary entry point for serving models. Let's trace the code:

```python
# sglang/launch_server.py (simplified)
"""
Usage:
    python -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct --port 30000
"""

import argparse
from sglang.srt.server import launch_server
from sglang.srt.server_args import ServerArgs

if __name__ == "__main__":
    # 1. Parse command-line arguments into ServerArgs
    server_args = ServerArgs.from_cli_args()
    
    # 2. Launch the server (blocking call)
    launch_server(server_args)
```

### 4.2 Python API: `sglang.api`

For programmatic usage within Python:

```python
# sglang/api.py (simplified)
import sglang as sgl

# Option 1: Use as a server
runtime = sgl.Runtime(model_path="meta-llama/Llama-3.1-8B-Instruct")
sgl.set_default_backend(runtime)

# Option 2: Use Engine directly (offline inference)
engine = sgl.Engine(model_path="meta-llama/Llama-3.1-8B-Instruct")
result = engine.generate("Hello, world!", max_new_tokens=32)
```

In [ ]:
# Source code walkthrough: launch_server.py
# Let's examine the actual launch flow

launch_server_code = '''
# === sglang/launch_server.py ===
# This is the main entry point when running:
#   python -m sglang.launch_server --model-path <model> --port <port>

import sys
from sglang.srt.server import launch_server
from sglang.srt.server_args import ServerArgs

def main():
    # Step 1: Parse CLI arguments
    server_args = ServerArgs.from_cli_args()  
    # ServerArgs contains ALL configuration:
    #   - model_path, tokenizer_path
    #   - port, host, api_key
    #   - tp_size (tensor parallelism)
    #   - dp_size (data parallelism)
    #   - mem_fraction_static (GPU memory fraction)
    #   - max_total_tokens, context_length
    #   - schedule_policy (lpm, random, fcfs, dfs-weight)
    #   - ... many more
    
    # Step 2: Launch the server
    launch_server(server_args)
    # This starts:
    #   (a) TokenizerManager process
    #   (b) Scheduler process(es)
    #   (c) DetokenizerManager process
    #   (d) FastAPI HTTP server
    #   (e) Optional: DataParallelController

if __name__ == "__main__":
    main()
'''

print(launch_server_code)

## 5. Configuration Classes

### 5.1 `ServerArgs` — Server Configuration

`ServerArgs` is the central configuration dataclass. It controls **everything** about how the server runs.

```python
# sglang/srt/server_args.py (key fields, simplified)

@dataclasses.dataclass
class ServerArgs:
    # === Model Configuration ===
    model_path: str                    # HuggingFace model ID or local path
    tokenizer_path: Optional[str]      # Custom tokenizer path
    dtype: str = "auto"                # Data type: auto, float16, bfloat16
    
    # === Server Configuration ===
    host: str = "127.0.0.1"            # Host to bind
    port: int = 30000                  # Port number
    api_key: Optional[str] = None      # API key for auth
    
    # === Parallelism ===
    tp_size: int = 1                   # Tensor parallel size
    dp_size: int = 1                   # Data parallel size  
    
    # === Memory Management ===
    mem_fraction_static: float = 0.88  # Fraction of GPU mem for KV-cache
    max_total_tokens: Optional[int]    # Max tokens in KV-cache pool
    context_length: Optional[int]      # Max context length
    
    # === Scheduling ===
    schedule_policy: str = "lpm"       # lpm, random, fcfs, dfs-weight
    schedule_conservativeness: float = 1.0
    chunk_prefill_budget: int = 0      # Chunked prefill tokens
    
    # === Attention Backend ===
    attention_backend: str = "flashinfer"  # flashinfer, triton
    
    # === Constrained Decoding ===
    grammar_backend: str = "xgrammar"  # xgrammar, outlines
    
    # === Performance ===
    disable_cuda_graph: bool = False
    cuda_graph_max_bs: int = 160
    enable_torch_compile: bool = False
    
    @classmethod
    def from_cli_args(cls):
        parser = argparse.ArgumentParser()
        # ... adds all fields as CLI args ...
        args = parser.parse_args()
        return cls(**vars(args))
```

In [ ]:
# Let's create a mock ServerArgs to understand its structure
import dataclasses
from typing import Optional, List

@dataclasses.dataclass
class ServerArgs:
    """Simplified SGLang ServerArgs for demonstration."""
    # Model
    model_path: str = "meta-llama/Llama-3.1-8B-Instruct"
    tokenizer_path: Optional[str] = None
    dtype: str = "auto"
    
    # Server
    host: str = "127.0.0.1"
    port: int = 30000
    api_key: Optional[str] = None
    
    # Parallelism
    tp_size: int = 1
    dp_size: int = 1
    
    # Memory
    mem_fraction_static: float = 0.88
    max_total_tokens: Optional[int] = None
    context_length: Optional[int] = None
    
    # Scheduling
    schedule_policy: str = "lpm"  # Longest Prefix Match
    chunk_prefill_budget: int = 0
    
    # Attention
    attention_backend: str = "flashinfer"
    
    # CUDA graphs
    disable_cuda_graph: bool = False
    cuda_graph_max_bs: int = 160

# Create default config
args = ServerArgs()
print("Default ServerArgs:")
print("=" * 50)
for field in dataclasses.fields(args):
    value = getattr(args, field.name)
    print(f"  {field.name:30s} = {value}")

print(f"\nTotal configuration fields: {len(dataclasses.fields(args))}")

### 5.2 `ModelConfig` — Model-Specific Configuration

While `ServerArgs` controls the server, `ModelConfig` wraps the model's HuggingFace configuration:

```python
# sglang/srt/configs/model_config.py (simplified)

class ModelConfig:
    def __init__(self, model_path, trust_remote_code=True):
        # Load HuggingFace config
        self.hf_config = AutoConfig.from_pretrained(model_path)
        
        # Extract key parameters
        self.vocab_size = self.hf_config.vocab_size
        self.hidden_size = self.hf_config.hidden_size
        self.num_hidden_layers = self.hf_config.num_hidden_layers
        self.num_attention_heads = self.hf_config.num_attention_heads
        self.num_key_value_heads = getattr(
            self.hf_config, "num_key_value_heads", 
            self.num_attention_heads  # default: MHA
        )
        
        # Derived parameters
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.is_multimodal = self._check_multimodal()
        self.context_length = self._get_context_length()
    
    def _check_multimodal(self):
        """Check if model supports images/video."""
        model_type = self.hf_config.model_type
        return model_type in ["llava", "qwen2_vl", "internvl", ...]
    
    def get_num_kv_heads(self, tp_size):
        """Get KV heads per TP rank."""
        return self.num_key_value_heads // tp_size
```

In [ ]:
# Simulated ModelConfig for common models

class ModelConfig:
    """Simplified ModelConfig for demonstration."""
    
    KNOWN_MODELS = {
        "meta-llama/Llama-3.1-8B-Instruct": {
            "vocab_size": 128256,
            "hidden_size": 4096,
            "num_hidden_layers": 32,
            "num_attention_heads": 32,
            "num_key_value_heads": 8,  # GQA
            "intermediate_size": 14336,
            "context_length": 131072,
            "model_type": "llama",
        },
        "Qwen/Qwen2.5-7B-Instruct": {
            "vocab_size": 152064,
            "hidden_size": 3584,
            "num_hidden_layers": 28,
            "num_attention_heads": 28,
            "num_key_value_heads": 4,  # GQA
            "intermediate_size": 18944,
            "context_length": 131072,
            "model_type": "qwen2",
        },
        "mistralai/Mixtral-8x7B-Instruct-v0.1": {
            "vocab_size": 32000,
            "hidden_size": 4096,
            "num_hidden_layers": 32,
            "num_attention_heads": 32,
            "num_key_value_heads": 8,  # GQA
            "intermediate_size": 14336,
            "context_length": 32768,
            "model_type": "mixtral",
            "num_experts": 8,
            "num_experts_per_tok": 2,
        },
    }
    
    def __init__(self, model_path):
        config = self.KNOWN_MODELS.get(model_path, {})
        for k, v in config.items():
            setattr(self, k, v)
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.is_multimodal = self.model_type in ["llava", "qwen2_vl"]
    
    def get_num_kv_heads(self, tp_size):
        return self.num_key_value_heads // tp_size
    
    def kv_cache_bytes_per_token(self):
        """Calculate KV-cache memory per token (in bytes, FP16)."""
        # 2 (K+V) * num_layers * num_kv_heads * head_dim * 2 (FP16 bytes)
        return 2 * self.num_hidden_layers * self.num_key_value_heads * self.head_dim * 2

# Compare models
for model_name in ModelConfig.KNOWN_MODELS:
    cfg = ModelConfig(model_name)
    kv_bytes = cfg.kv_cache_bytes_per_token()
    print(f"\n{model_name}")
    print(f"  Attention heads: {cfg.num_attention_heads} Q / {cfg.num_key_value_heads} KV (GQA ratio: {cfg.num_attention_heads // cfg.num_key_value_heads}x)")
    print(f"  Head dim: {cfg.head_dim}")
    print(f"  KV-cache per token: {kv_bytes:,} bytes ({kv_bytes / 1024:.1f} KB)")
    print(f"  Max context: {cfg.context_length:,} tokens")
    print(f"  KV-cache for full context: {kv_bytes * cfg.context_length / 1024**3:.2f} GB")

## 6. Build System and Dependencies

### 6.1 Installation Methods

SGLang can be installed in several ways:

```bash
# Method 1: pip install (recommended)
pip install "sglang[all]"

# Method 2: Install with specific backends
pip install "sglang[srt]"    # Runtime only
pip install "sglang[openai]" # OpenAI backend only

# Method 3: From source
git clone https://github.com/sgl-project/sglang.git
cd sglang
pip install -e "python[all]"

# Method 4: Docker
docker run --gpus all -p 30000:30000 \
    lmsysorg/sglang:latest \
    python -m sglang.launch_server --model-path meta-llama/Llama-3.1-8B-Instruct
```

### 6.2 Key Dependencies

| Dependency | Purpose | Version |
|------------|---------|----------|
| `torch` | Core tensor operations | >= 2.3.0 |
| `flashinfer` | High-performance attention kernels | >= 0.1.6 |
| `vllm` | Model loading, weight management | >= 0.6.0 |
| `transformers` | Tokenizers, model configs | >= 4.40.0 |
| `fastapi` | HTTP server framework | >= 0.110.0 |
| `uvicorn` | ASGI server | >= 0.28.0 |
| `triton` | Triton kernel backend | >= 2.3.0 |
| `xgrammar` | Grammar-guided decoding | >= 0.1.0 |
| `outlines` | Alternative constrained decoding | >= 0.0.44 |
| `zmq` | Inter-process communication | >= 25.0 |

**Important note**: SGLang actually **depends on vLLM** for model weight loading and some model implementations. The two projects share model code infrastructure.

In [ ]:
# Check which SGLang dependencies are available
dependencies = [
    ("torch", "PyTorch"),
    ("transformers", "HuggingFace Transformers"),
    ("fastapi", "FastAPI"),
    ("uvicorn", "Uvicorn"),
    ("zmq", "ZeroMQ"),
    ("triton", "Triton"),
    ("sglang", "SGLang"),
    ("vllm", "vLLM"),
]

print("SGLang Dependency Status")
print("=" * 60)
for module_name, description in dependencies:
    try:
        mod = __import__(module_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  [OK] {description:30s} ({module_name} {version})")
    except ImportError:
        print(f"  [--] {description:30s} ({module_name} not installed)")

## 7. Server Startup Flow (Detailed)

When you run `python -m sglang.launch_server`, here's what happens step by step:

```
┌─────────────────────────────────────────────────────────────────┐
│                    Server Startup Flow                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. Parse CLI args → ServerArgs                                │
│     │                                                           │
│  2. launch_server(server_args)                                 │
│     │                                                           │
│  3. Create inter-process communication channels (ZMQ)          │
│     │                                                           │
│  4. Fork processes:                                            │
│     │                                                           │
│     ├── Process 1: TokenizerManager                            │
│     │   └── Loads tokenizer, handles encode/decode             │
│     │                                                           │
│     ├── Process 2: Scheduler                                   │
│     │   ├── Initializes TpModelWorker(s)                       │
│     │   │   └── Each worker loads model on GPU                 │
│     │   ├── Creates RadixCache                                 │
│     │   ├── Creates MemoryPool                                 │
│     │   └── Starts scheduling loop                             │
│     │                                                           │
│     ├── Process 3: DetokenizerManager                          │
│     │   └── Handles incremental detokenization                 │
│     │                                                           │
│     └── (Optional) DataParallelController                      │
│         └── Routes requests across DP replicas                 │
│                                                                 │
│  5. Start FastAPI HTTP server (uvicorn)                        │
│     ├── /v1/chat/completions                                   │
│     ├── /v1/completions                                        │
│     ├── /generate                                              │
│     └── /health                                                │
│                                                                 │
│  6. Server is ready! Print URL                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Detailed source code walkthrough: server.py launch flow

server_launch_code = '''
# === sglang/srt/server.py (simplified) ===

def launch_server(server_args: ServerArgs):
    """Main entry point to launch the SGLang server."""
    
    # Step 1: Set up logging
    configure_logger(server_args)
    
    # Step 2: Validate server args
    server_args.check_server_args()
    
    # Step 3: Create port for inter-process communication
    port_args = PortArgs.init_new(server_args)
    # port_args contains ZMQ socket addresses for:
    #   - tokenizer_ipc_name: TokenizerManager <-> Scheduler
    #   - scheduler_input_ipc_name: input to scheduler
    #   - scheduler_output_ipc_name: output from scheduler
    #   - detokenizer_ipc_name: Scheduler <-> DetokenizerManager
    
    # Step 4: Launch child processes
    # 4a. Launch Scheduler process(es)
    scheduler_procs = []
    for dp_rank in range(server_args.dp_size):
        proc = multiprocessing.Process(
            target=start_scheduler_process,
            args=(server_args, port_args, dp_rank)
        )
        proc.start()
        scheduler_procs.append(proc)
    
    # 4b. Launch TokenizerManager process
    tokenizer_proc = multiprocessing.Process(
        target=start_tokenizer_process,
        args=(server_args, port_args)
    )
    tokenizer_proc.start()
    
    # 4c. Launch DetokenizerManager process
    detokenizer_proc = multiprocessing.Process(
        target=start_detokenizer_process,
        args=(server_args, port_args)
    )
    detokenizer_proc.start()
    
    # Step 5: Wait for all processes to be ready
    wait_for_ready(scheduler_procs, tokenizer_proc, detokenizer_proc)
    
    # Step 6: Create and start FastAPI app
    app = create_app(server_args, port_args)
    uvicorn.run(app, host=server_args.host, port=server_args.port)
'''

print(server_launch_code)

## 8. Inter-Process Communication (IPC)

SGLang uses **ZeroMQ (ZMQ)** for communication between processes. This is a key architectural decision:

```
┌──────────────┐     ZMQ      ┌──────────────┐     ZMQ      ┌──────────────┐
│  Tokenizer   │────────────→ │  Scheduler    │────────────→ │ Detokenizer  │
│  Manager     │              │               │              │ Manager      │
│              │              │  ┌──────────┐ │              │              │
│  - tokenize  │              │  │ RadixCache│ │              │  - detokenize│
│  - manage    │              │  │ MemPool   │ │              │  - stream    │
│    requests  │              │  │ TpWorker  │ │              │    output    │
└──────────────┘              │  └──────────┘ │              └──────────────┘
       ↑                      └──────────────┘                      │
       │                                                            │
       │              ┌──────────────┐                              │
       └──────────────│  FastAPI     │──────────────────────────────┘
                      │  HTTP Server │
                      │  (uvicorn)   │
                      └──────────────┘
                             ↑
                        HTTP requests
```

### Why ZMQ instead of Python multiprocessing Queues?

1. **Performance**: ZMQ is faster for message passing (no GIL contention)
2. **Flexibility**: Supports TCP, IPC, and in-process transports
3. **Scalability**: Easy to distribute across machines for multi-node
4. **Patterns**: Built-in pub/sub, push/pull, request/reply patterns

In [ ]:
# Demonstration of ZMQ-style inter-process communication
# (Simplified version of SGLang's IPC)

import json
import threading
import queue
import time

class SimulatedZMQChannel:
    """Simulates ZMQ inter-process communication."""
    def __init__(self, name):
        self.name = name
        self.queue = queue.Queue()
    
    def send(self, message):
        self.queue.put(json.dumps(message))
    
    def recv(self, timeout=1.0):
        try:
            return json.loads(self.queue.get(timeout=timeout))
        except queue.Empty:
            return None

# Simulate SGLang's process architecture
tokenizer_to_scheduler = SimulatedZMQChannel("tokenizer->scheduler")
scheduler_to_detokenizer = SimulatedZMQChannel("scheduler->detokenizer")

def tokenizer_process(request_text, channel_out):
    """Simulates TokenizerManager."""
    # Tokenize the request
    tokens = list(range(len(request_text.split())))  # fake token IDs
    message = {
        "type": "tokenized_request",
        "token_ids": tokens,
        "original_text": request_text,
    }
    print(f"[TokenizerManager] Tokenized: '{request_text}' -> {tokens}")
    channel_out.send(message)

def scheduler_process(channel_in, channel_out):
    """Simulates Scheduler."""
    msg = channel_in.recv()
    if msg:
        # Schedule and run model forward pass
        output_tokens = msg["token_ids"] + [42, 43, 44]  # fake generation
        result = {
            "type": "generation_result",
            "output_token_ids": output_tokens,
        }
        print(f"[Scheduler] Generated output tokens: {output_tokens}")
        channel_out.send(result)

def detokenizer_process(channel_in):
    """Simulates DetokenizerManager."""
    msg = channel_in.recv()
    if msg:
        # Detokenize
        text = f"Generated text with {len(msg['output_token_ids'])} tokens"
        print(f"[DetokenizerManager] Final output: '{text}'")
        return text

# Run the pipeline
print("=== Simulated SGLang Request Pipeline ===")
print()

# Step 1: Tokenize
tokenizer_process("Hello, how are you?", tokenizer_to_scheduler)

# Step 2: Schedule & Generate
scheduler_process(tokenizer_to_scheduler, scheduler_to_detokenizer)

# Step 3: Detokenize
detokenizer_process(scheduler_to_detokenizer)

print("\n=== Pipeline complete ===")

## 9. Dependency Graph Between Components

Let's visualize how SGLang's components depend on each other:

In [ ]:
# Build a dependency graph visualization

dependency_graph = {
    # Top-level entry points
    "launch_server.py": ["ServerArgs", "server.py"],
    "api.py": ["Runtime", "Engine"],
    
    # Server layer
    "server.py": ["TokenizerManager", "Scheduler", "DetokenizerManager", "FastAPI"],
    
    # Manager layer
    "TokenizerManager": ["HF Tokenizer", "ZMQ"],
    "Scheduler": ["RadixCache", "MemoryPool", "TpModelWorker", "SchedulePolicy"],
    "DetokenizerManager": ["HF Tokenizer", "ZMQ"],
    
    # Worker layer
    "TpModelWorker": ["ModelRunner"],
    "ModelRunner": ["Model", "FlashInfer", "CUDAGraph", "InputMetadata"],
    
    # Cache layer
    "RadixCache": ["RadixTree", "MemoryPool"],
    "MemoryPool": ["CUDA Memory"],
    
    # Model layer
    "Model": ["Attention", "MLP", "Embedding", "LayerNorm"],
    "Attention": ["FlashInfer", "Triton"],
}

def print_dependency_tree(graph, node, visited=None, prefix="", is_last=True):
    """Print dependency tree."""
    if visited is None:
        visited = set()
    
    connector = "└── " if is_last else "├── "
    print(f"{prefix}{connector}{node}")
    
    if node in visited:
        return
    visited.add(node)
    
    children = graph.get(node, [])
    for i, child in enumerate(children):
        extension = "    " if is_last else "│   "
        print_dependency_tree(
            graph, child, visited.copy(),
            prefix + extension, i == len(children) - 1
        )

print("SGLang Component Dependency Tree")
print("=" * 60)
print("\nFrom launch_server.py:")
print_dependency_tree(dependency_graph, "launch_server.py", prefix="")

print("\nFrom api.py:")
print_dependency_tree(dependency_graph, "api.py", prefix="")

## 10. SGLang vs vLLM: Design Philosophy Comparison

### 10.1 Architecture Comparison

```
vLLM Architecture:                    SGLang Architecture:
┌─────────────────┐                   ┌─────────────────┐
│   AsyncEngine   │                   │   HTTP Server    │
│  (single class) │                   │   (FastAPI)      │
├─────────────────┤                   ├─────────────────┤
│   Scheduler     │                   │ TokenizerMgr    │
│   (in-process)  │                   │ (separate proc) │
├─────────────────┤                   ├─────────────────┤
│   PagedAttention│                   │ Scheduler       │
│   BlockManager  │                   │ RadixCache      │
├─────────────────┤                   │ (separate proc) │
│   ModelRunner   │                   ├─────────────────┤
│   Worker(s)     │                   │ DetokenizerMgr  │
└─────────────────┘                   │ (separate proc) │
                                      ├─────────────────┤
                                      │ TpModelWorker   │
                                      │ ModelRunner     │
                                      └─────────────────┘
```

### 10.2 Key Design Differences

| Aspect | vLLM | SGLang |
|--------|------|--------|
| **Process Model** | Single-process with async | Multi-process with ZMQ |
| **KV-Cache** | Block-based (PagedAttention) | Tree-based (RadixAttention) |
| **Prefix Caching** | Opt-in, hash-based | Automatic via radix tree |
| **Scheduling** | Priority queue + preemption | LPM (Longest Prefix Match) |
| **Frontend** | OpenAI API | DSL + OpenAI API |
| **Attention** | Multiple backends | FlashInfer-first |
| **Constrained Gen** | Via plugins | Native support |
| **Model Loading** | Custom loader | Reuses vLLM's loader |
| **Tokenization** | In-process | Separate process |

In [ ]:
# Detailed comparison of KV-cache strategies

print("KV-Cache Strategy Comparison")
print("=" * 70)

comparison = {
    "Data Structure": (
        "Hash table of block sequences",
        "Radix tree (Patricia trie)"
    ),
    "Prefix Sharing": (
        "Hash-based matching (opt-in)",
        "Automatic via tree structure"
    ),
    "Granularity": (
        "Block-level (typically 16 tokens)",
        "Token-level with page backing"
    ),
    "Eviction Policy": (
        "Preemption (swap/recompute)",
        "LRU eviction of tree nodes"
    ),
    "Multi-turn Reuse": (
        "Only with prefix caching enabled",
        "Automatic (tree retains history)"
    ),
    "System Prompt Cache": (
        "Requires explicit APC setup",
        "Automatic (shared prefix in tree)"
    ),
    "Memory Overhead": (
        "Low (simple block table)",
        "Moderate (tree node metadata)"
    ),
    "Lookup Speed": (
        "O(1) per block hash",
        "O(n) tree traversal, but amortized"
    ),
}

print(f"{'Aspect':<25s} {'vLLM (PagedAttention)':<35s} {'SGLang (RadixAttention)'}")
print("-" * 95)
for aspect, (vllm_val, sglang_val) in comparison.items():
    print(f"{aspect:<25s} {vllm_val:<35s} {sglang_val}")

## 11. Key Source Files Reference

Here's a quick reference of the most important source files in SGLang:

### Runtime (`srt/`)

| File | Purpose | Lines (approx) |
|------|---------|----------------|
| `server.py` | FastAPI server, launch logic | ~500 |
| `server_args.py` | All server configuration | ~400 |
| `managers/scheduler.py` | Core scheduling logic | ~1200 |
| `managers/tokenizer_manager.py` | Tokenization management | ~300 |
| `managers/detokenizer_manager.py` | Detokenization | ~250 |
| `managers/tp_worker.py` | Tensor parallel worker | ~400 |
| `managers/io_struct.py` | Request/Response data classes | ~300 |
| `mem_cache/radix_cache.py` | RadixAttention radix tree | ~600 |
| `mem_cache/memory_pool.py` | GPU memory pool management | ~400 |
| `model_executor/model_runner.py` | Forward pass execution | ~800 |
| `model_executor/forward_batch_info.py` | Batch metadata | ~500 |
| `layers/attention/flashinfer_backend.py` | FlashInfer integration | ~400 |
| `constrained/xgrammar_backend.py` | Grammar-guided decoding | ~300 |

### Frontend (`lang/`)

| File | Purpose | Lines (approx) |
|------|---------|----------------|
| `ir.py` | DSL intermediate representation | ~300 |
| `interpreter.py` | Program interpreter | ~500 |
| `compiler.py` | Optimized compilation | ~200 |
| `backend/runtime_endpoint.py` | SRT backend connector | ~300 |

In [ ]:
# Summary of SGLang's architecture in code

class SGLangArchitecture:
    """A conceptual model of SGLang's architecture."""
    
    LAYERS = {
        "Entry Points": [
            "launch_server.py  - CLI entry point",
            "api.py            - Python API (Runtime, Engine)",
        ],
        "API Layer": [
            "FastAPI server     - OpenAI-compatible HTTP API",
            "SGLang DSL         - Structured generation language",
        ],
        "Processing Layer": [
            "TokenizerManager   - Concurrent tokenization",
            "DetokenizerManager - Incremental detokenization",
        ],
        "Scheduling Layer": [
            "Scheduler          - Request scheduling (LPM, FCFS)",
            "RadixCache         - Radix tree KV-cache management",
            "MemoryPool         - GPU memory allocation",
        ],
        "Execution Layer": [
            "TpModelWorker      - Tensor parallel coordination",
            "ModelRunner        - Forward pass execution",
            "CUDAGraphRunner    - CUDA graph capture/replay",
        ],
        "Kernel Layer": [
            "FlashInfer         - Attention kernels",
            "Triton kernels     - Custom GPU kernels",
            "XGrammar           - Constrained decoding FSM",
        ],
    }
    
    @classmethod
    def display(cls):
        print("\nSGLang Architecture Layers")
        print("=" * 60)
        for layer_name, components in cls.LAYERS.items():
            print(f"\n{'─' * 60}")
            print(f"  {layer_name}")
            print(f"{'─' * 60}")
            for component in components:
                print(f"    {component}")
        print(f"\n{'=' * 60}")

SGLangArchitecture.display()

## 12. The `__init__.py` — Public API Surface

SGLang's `__init__.py` exports the public API:

```python
# sglang/__init__.py (simplified)

# Frontend DSL primitives
from sglang.lang.ir import (
    SglFunction as function,
    gen,
    select,
    system,
    user,
    assistant,
    image,
    video,
)

# Runtime
from sglang.api import (
    Runtime,
    Engine,
    set_default_backend,
    get_default_backend,
)

# Backend connectors
from sglang.lang.backend import (
    RuntimeEndpoint,
    OpenAI,
    Anthropic,
)
```

This gives users a clean API:
```python
import sglang as sgl

@sgl.function
def my_program(s, question):
    s += sgl.user(question)
    s += sgl.assistant(sgl.gen("answer"))
```

In [ ]:
# Quick reference: all CLI arguments for launch_server

cli_args = [
    ("--model-path", "str", "REQUIRED", "HuggingFace model ID or local path"),
    ("--tokenizer-path", "str", "None", "Custom tokenizer path"),
    ("--host", "str", "127.0.0.1", "Host to bind the server"),
    ("--port", "int", "30000", "Port number"),
    ("--tp-size", "int", "1", "Tensor parallelism degree"),
    ("--dp-size", "int", "1", "Data parallelism degree"),
    ("--mem-fraction-static", "float", "0.88", "GPU memory fraction for KV-cache"),
    ("--max-total-tokens", "int", "auto", "Max tokens in KV-cache"),
    ("--context-length", "int", "auto", "Max context length"),
    ("--schedule-policy", "str", "lpm", "Scheduling policy: lpm/random/fcfs"),
    ("--attention-backend", "str", "flashinfer", "Attention backend"),
    ("--grammar-backend", "str", "xgrammar", "Grammar backend for constrained gen"),
    ("--disable-cuda-graph", "flag", "False", "Disable CUDA graph optimization"),
    ("--dtype", "str", "auto", "Model data type (float16/bfloat16)"),
    ("--quantization", "str", "None", "Quantization method (awq/gptq/fp8)"),
    ("--api-key", "str", "None", "API key for authentication"),
    ("--chat-template", "str", "None", "Custom chat template"),
    ("--trust-remote-code", "flag", "False", "Trust remote code from HF"),
]

print("SGLang Server CLI Arguments")
print("=" * 90)
print(f"{'Argument':<30s} {'Type':<8s} {'Default':<12s} {'Description'}")
print("-" * 90)
for arg, typ, default, desc in cli_args:
    print(f"{arg:<30s} {typ:<8s} {default:<12s} {desc}")

print(f"\nExample:")
print("  python -m sglang.launch_server \\")
print("    --model-path meta-llama/Llama-3.1-8B-Instruct \\")
print("    --port 30000 \\")
print("    --tp-size 1 \\")
print("    --mem-fraction-static 0.9")

## 13. Data Flow: Request Lifecycle

Let's trace a complete request through SGLang:

```
HTTP Request (POST /v1/chat/completions)
    │
    ▼
┌─ FastAPI Handler ──────────────────────────────────────────────┐
│  1. Parse OpenAI-format request                                │
│  2. Create GenerateReqInput (internal request object)          │
│  3. Send to TokenizerManager via ZMQ                           │
└────────────────────────────────────────────────────────────────┘
    │
    ▼
┌─ TokenizerManager ─────────────────────────────────────────────┐
│  4. Tokenize input text (apply chat template if needed)        │
│  5. Create TokenizedGenerateReqInput with token IDs            │
│  6. Send to Scheduler via ZMQ                                  │
└────────────────────────────────────────────────────────────────┘
    │
    ▼
┌─ Scheduler ────────────────────────────────────────────────────┐
│  7. Check RadixCache for prefix matches                        │
│  8. Allocate memory from MemoryPool                            │
│  9. Add to waiting queue                                       │
│ 10. Build batch (extend/decode) based on schedule policy       │
│ 11. Send batch to TpModelWorker                                │
└────────────────────────────────────────────────────────────────┘
    │
    ▼
┌─ TpModelWorker / ModelRunner ──────────────────────────────────┐
│ 12. Prepare input tensors (token IDs, positions, metadata)     │
│ 13. Run model forward pass (FlashInfer attention)              │
│ 14. Sample next token(s)                                       │
│ 15. Return output to Scheduler                                 │
└────────────────────────────────────────────────────────────────┘
    │
    ▼
┌─ Scheduler (continued) ────────────────────────────────────────┐
│ 16. Update RadixCache with new tokens                          │
│ 17. Check stopping criteria (EOS, max tokens)                  │
│ 18. If not done, add to next decode batch → go to step 10     │
│ 19. If done, send output tokens to DetokenizerManager          │
└────────────────────────────────────────────────────────────────┘
    │
    ▼
┌─ DetokenizerManager ───────────────────────────────────────────┐
│ 20. Incrementally detokenize output tokens                     │
│ 21. Send text back to FastAPI handler                          │
└────────────────────────────────────────────────────────────────┘
    │
    ▼
HTTP Response (SSE stream or JSON)
```

In [ ]:
# Complete request lifecycle simulation

import time
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class GenerateReqInput:
    """Mirrors sglang.srt.managers.io_struct.GenerateReqInput"""
    text: str
    sampling_params: dict
    rid: str  # request ID

@dataclass  
class TokenizedRequest:
    """Internal tokenized request."""
    rid: str
    input_ids: List[int]
    sampling_params: dict
    prefix_len: int = 0  # how much matches in RadixCache

@dataclass
class BatchOutput:
    """Output from model forward pass."""
    rid: str
    output_ids: List[int]
    finished: bool

class RequestLifecycle:
    """Traces a request through SGLang's pipeline."""
    
    def __init__(self):
        self.steps = []
    
    def log(self, component, action, details=""):
        self.steps.append((component, action, details))
    
    def trace_request(self, text, max_tokens=10):
        """Simulate a full request lifecycle."""
        rid = "req-001"
        
        # Step 1: HTTP handler
        self.log("FastAPI", "Receive request", f"text='{text[:30]}...'")
        req = GenerateReqInput(text=text, sampling_params={"max_tokens": max_tokens}, rid=rid)
        
        # Step 2: TokenizerManager
        self.log("TokenizerManager", "Tokenize input", f"text -> {len(text.split())} tokens (simulated)")
        token_ids = list(range(len(text.split())))  # fake
        tok_req = TokenizedRequest(rid=rid, input_ids=token_ids, sampling_params=req.sampling_params)
        
        # Step 3: Scheduler - check cache
        prefix_match = min(3, len(token_ids))  # simulate partial match
        self.log("Scheduler", "RadixCache lookup", f"prefix match: {prefix_match}/{len(token_ids)} tokens")
        tok_req.prefix_len = prefix_match
        
        # Step 4: Scheduler - allocate memory
        new_tokens_to_process = len(token_ids) - prefix_match
        self.log("Scheduler", "Allocate memory", f"{new_tokens_to_process} new tokens to prefill")
        
        # Step 5: Extend (prefill) phase
        self.log("ModelRunner", "Extend forward pass", f"processing {new_tokens_to_process} tokens")
        
        # Step 6: Decode loop
        generated = []
        for i in range(max_tokens):
            new_token = 1000 + i  # fake token
            generated.append(new_token)
            is_eos = (i == max_tokens - 1)
            self.log("ModelRunner", f"Decode step {i+1}", 
                     f"token={new_token}" + (" [EOS]" if is_eos else ""))
            
            if is_eos:
                break
        
        # Step 7: Update cache
        self.log("Scheduler", "Update RadixCache", f"insert {len(token_ids) + len(generated)} tokens")
        
        # Step 8: Detokenize
        self.log("DetokenizerManager", "Detokenize output", f"{len(generated)} tokens -> text")
        
        # Step 9: Return response
        self.log("FastAPI", "Send response", "HTTP 200 OK")
    
    def display(self):
        print("\nRequest Lifecycle Trace")
        print("=" * 80)
        for i, (component, action, details) in enumerate(self.steps):
            print(f"  [{i+1:2d}] {component:<22s} | {action:<25s} | {details}")

# Run trace
lifecycle = RequestLifecycle()
lifecycle.trace_request("What is the capital of France? Please explain.", max_tokens=5)
lifecycle.display()

## 14. I/O Data Structures

SGLang defines several key data structures for passing data between components:

```python
# sglang/srt/managers/io_struct.py (key classes)

@dataclass
class GenerateReqInput:
    """Input from HTTP API to TokenizerManager."""
    text: Optional[str] = None       # Raw text input
    input_ids: Optional[List] = None # Pre-tokenized input
    sampling_params: dict = None
    rid: str = None                  # Request ID
    stream: bool = False             # Streaming response
    
@dataclass
class TokenizedGenerateReqInput:
    """Tokenized request from TokenizerManager to Scheduler."""
    rid: str
    input_ids: List[int]
    pixel_values: Optional[torch.Tensor]  # For multimodal
    sampling_params: SamplingParams
    
@dataclass
class BatchTokenIDOut:
    """Output from Scheduler to DetokenizerManager."""
    rids: List[str]           # Request IDs in batch
    output_ids: List[List[int]] # Generated token IDs
    finished: List[bool]      # Which requests are done
    
@dataclass
class BatchStrOut:
    """Final output from DetokenizerManager."""
    rids: List[str]
    output_strs: List[str]    # Generated text
    finished: List[bool]
```

In [ ]:
# Demonstrate the I/O data flow with concrete examples

print("I/O Data Flow Example")
print("=" * 60)

# 1. HTTP request comes in
http_request = {
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [
        {"role": "user", "content": "What is 2+2?"}
    ],
    "max_tokens": 50,
    "temperature": 0.7,
}
print("\n1. HTTP Request (OpenAI format):")
print(f"   {http_request}")

# 2. Converted to GenerateReqInput
generate_req = {
    "rid": "req-abc123",
    "text": "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat is 2+2?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    "sampling_params": {"max_new_tokens": 50, "temperature": 0.7},
}
print("\n2. GenerateReqInput (after chat template):")
print(f"   rid: {generate_req['rid']}")
print(f"   text: '{generate_req['text'][:50]}...'")

# 3. Tokenized
tokenized = {
    "rid": "req-abc123",
    "input_ids": [128000, 128006, 882, 128007, 271, 3923, 374, 220, 17, 10, 17, 30, 128009, 128006, 78191, 128007, 271],
    "sampling_params": {"max_new_tokens": 50, "temperature": 0.7},
}
print(f"\n3. TokenizedGenerateReqInput:")
print(f"   input_ids: {tokenized['input_ids']}")
print(f"   length: {len(tokenized['input_ids'])} tokens")

# 4. Scheduler output
batch_output = {
    "rids": ["req-abc123"],
    "output_ids": [[17, 10, 17, 284, 220, 19, 13]],  # "2+2 = 4."
    "finished": [True],
}
print(f"\n4. BatchTokenIDOut:")
print(f"   output_ids: {batch_output['output_ids']}")

# 5. Detokenized
final_output = {
    "rids": ["req-abc123"],
    "output_strs": ["2+2 = 4."],
    "finished": [True],
}
print(f"\n5. BatchStrOut:")
print(f"   output: '{final_output['output_strs'][0]}'")

## 15. Model Registry and Support

SGLang supports a wide range of models. The model registry maps HuggingFace model types to SGLang model implementations:

```python
# sglang/srt/models/model_registry.py (simplified)

MODEL_REGISTRY = {
    # Text-only models
    "LlamaForCausalLM": "sglang.srt.models.llama",
    "Qwen2ForCausalLM": "sglang.srt.models.qwen2",
    "MistralForCausalLM": "sglang.srt.models.llama",  # same arch
    "MixtralForCausalLM": "sglang.srt.models.mixtral",
    "GemmaForCausalLM": "sglang.srt.models.gemma",
    "Gemma2ForCausalLM": "sglang.srt.models.gemma2",
    "Phi3ForCausalLM": "sglang.srt.models.llama",
    "DeepseekV2ForCausalLM": "sglang.srt.models.deepseek_v2",
    "DeepseekV3ForCausalLM": "sglang.srt.models.deepseek_v3",
    
    # Multi-modal models
    "LlavaForConditionalGeneration": "sglang.srt.models.llava",
    "Qwen2VLForConditionalGeneration": "sglang.srt.models.qwen2_vl",
    "InternVLChatModel": "sglang.srt.models.internvl",
    
    # Embedding models
    "MistralModel": "sglang.srt.models.llama_embedding",
}
```

Many SGLang model implementations **borrow from or directly use vLLM's model code**, especially for weight loading.

In [ ]:
# Model support summary

supported_models = {
    "Text Models": [
        ("Llama 3/3.1/3.2", "llama.py", "Meta"),
        ("Qwen 2/2.5", "qwen2.py", "Alibaba"),
        ("Mistral/Mixtral", "mixtral.py", "Mistral AI"),
        ("Gemma/Gemma2", "gemma2.py", "Google"),
        ("Phi-3/3.5", "llama.py", "Microsoft"),
        ("DeepSeek V2/V3", "deepseek_v2.py", "DeepSeek"),
        ("Yi", "llama.py", "01.AI"),
        ("Command-R", "commandr.py", "Cohere"),
        ("Starcoder2", "starcoder2.py", "BigCode"),
    ],
    "Multi-Modal Models": [
        ("LLaVA 1.5/1.6/NeXT", "llava.py", "LLaVA Team"),
        ("Qwen-VL / Qwen2-VL", "qwen2_vl.py", "Alibaba"),
        ("InternVL 1.5/2", "internvl.py", "Shanghai AI Lab"),
        ("Llama 3.2 Vision", "mllama.py", "Meta"),
    ],
    "Embedding Models": [
        ("E5-Mistral", "llama_embedding.py", "Microsoft"),
        ("GTE", "qwen2_embedding.py", "Alibaba"),
    ],
}

print("SGLang Supported Models")
print("=" * 70)
for category, models in supported_models.items():
    print(f"\n  {category}:")
    print(f"  {'Model':<30s} {'Implementation':<25s} {'Provider'}")
    print(f"  {'-'*65}")
    for model, impl, provider in models:
        print(f"  {model:<30s} {impl:<25s} {provider}")

## 16. Quick Start Demo

Here's how to get started with SGLang in practice:

In [ ]:
# Demo 1: Starting a server (you'd run this in a terminal)

print("=== Quick Start: Launching SGLang Server ===")
print()
print("# Terminal 1: Start the server")
print("python -m sglang.launch_server \\")
print("    --model-path meta-llama/Llama-3.1-8B-Instruct \\")
print("    --port 30000 \\")
print("    --tp-size 1")
print()
print("# Terminal 2: Send a request with curl")
print("""curl -X POST http://localhost:30000/v1/chat/completions \\""")
print("""  -H "Content-Type: application/json" \\""")
print("""  -d '{
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [{"role": "user", "content": "Hello!"}],
    "max_tokens": 50
  }'""")
print()
print("# Python client")
print("""import openai
client = openai.Client(base_url="http://localhost:30000/v1", api_key="none")
response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}],
    max_tokens=50
)
print(response.choices[0].message.content)""")

In [ ]:
# Demo 2: Using SGLang's offline Engine API

print("=== Quick Start: Offline Engine API ===")
print()
print("""import sglang as sgl

# Create an engine (loads model, no HTTP server)
engine = sgl.Engine(model_path="meta-llama/Llama-3.1-8B-Instruct")

# Generate text
result = engine.generate(
    prompt="The capital of France is",
    sampling_params={"max_new_tokens": 20, "temperature": 0}
)
print(result["text"])

# Batch generation
prompts = [
    "The capital of France is",
    "The largest planet is",
    "Water boils at",
]
results = engine.generate(prompts, sampling_params={"max_new_tokens": 20})
for prompt, result in zip(prompts, results):
    print(f"{prompt} -> {result['text']}")

# Clean up
engine.shutdown()
""")

In [ ]:
# Demo 3: Using SGLang's DSL

print("=== Quick Start: SGLang DSL ===")
print()
print("""import sglang as sgl

# Define a structured LLM program
@sgl.function
def analyze_and_summarize(s, text):
    s += sgl.system("You are a helpful assistant.")
    s += sgl.user(f"Analyze the following text: {text}")
    s += sgl.assistant(sgl.gen("analysis", max_tokens=200))
    s += sgl.user("Now provide a one-sentence summary.")
    s += sgl.assistant(sgl.gen("summary", max_tokens=50))

# Set up backend
runtime = sgl.Runtime(model_path="meta-llama/Llama-3.1-8B-Instruct")
sgl.set_default_backend(runtime)

# Run the program
state = analyze_and_summarize.run(text="SGLang is a fast inference framework...")

# Access named outputs
print("Analysis:", state["analysis"])
print("Summary:", state["summary"])

# Batch execution (processes multiple inputs in parallel)
states = analyze_and_summarize.run_batch([
    {"text": "First document..."},
    {"text": "Second document..."},
    {"text": "Third document..."},
])

runtime.shutdown()
""")

## 17. Summary

### Key Takeaways

1. **SGLang has two main components**: `srt` (runtime) and `lang` (frontend DSL)
2. **Multi-process architecture**: Tokenizer, Scheduler, and Detokenizer run in separate processes connected via ZMQ
3. **RadixAttention** is the core innovation: automatic prefix caching via radix tree
4. **ServerArgs** is the central configuration with 50+ parameters
5. **ModelConfig** wraps HuggingFace configs for runtime use
6. **Entry points**: `launch_server.py` (CLI) and `api.py` (Python API)
7. **SGLang depends on vLLM** for model weight loading infrastructure
8. **FlashInfer is the primary attention backend**, unlike vLLM which uses FlashAttention

### What's Next

In the following chapters, we'll dive deep into each component:
- **Chapter 5.2**: Runtime server architecture and request routing
- **Chapter 5.3**: RadixAttention and the radix tree data structure
- **Chapter 5.4**: Scheduler internals and scheduling policies

## Exercises

### Exercise 1: Codebase Navigation
Clone the SGLang repository and answer these questions:
1. How many model implementations are in `sglang/srt/models/`?
2. What is the default `schedule_policy` in `ServerArgs`?
3. Find the `RadixCache` class. How many methods does it have?

### Exercise 2: Configuration Exploration
Create a `ServerArgs` instance with the following settings:
- Llama 3.1 70B model
- 4-GPU tensor parallelism
- 90% GPU memory for KV-cache
- Chunked prefill with 4096-token budget

### Exercise 3: Architecture Diagram
Draw a complete architecture diagram of SGLang showing:
- All processes and their responsibilities
- Communication channels between processes
- Data structures at each boundary

### Exercise 4: Comparison Essay
Write a 500-word comparison of SGLang vs vLLM covering:
- KV-cache management approach
- Process model and IPC
- Frontend API design
- Which is better suited for which use cases?

In [ ]:
# Exercise 2 starter code

# TODO: Fill in the correct values
exercise_args = ServerArgs(
    model_path="meta-llama/Llama-3.1-70B-Instruct",
    tp_size=4,              # 4-GPU tensor parallelism
    mem_fraction_static=0.90,  # 90% GPU memory for KV-cache
    # chunk_prefill_budget=???  # What should this be?
)

print("Exercise 2 - Your ServerArgs:")
for field in dataclasses.fields(exercise_args):
    print(f"  {field.name}: {getattr(exercise_args, field.name)}")

# Bonus: Calculate expected KV-cache capacity
# Llama 3.1 70B: 80 layers, 8 KV heads, head_dim=128
kv_per_token = 2 * 80 * 8 * 128 * 2  # bytes (FP16)
gpu_mem_gb = 80  # A100 80GB
total_gpu_mem = gpu_mem_gb * exercise_args.tp_size  # GB across all GPUs
kv_mem = total_gpu_mem * exercise_args.mem_fraction_static
# But model weights also need memory...
model_size_gb = 140  # ~140GB for 70B FP16
available_for_kv = total_gpu_mem - model_size_gb  # rough estimate
max_tokens = int(available_for_kv * 1024**3 / kv_per_token)
print(f"\nEstimated KV-cache capacity: ~{max_tokens:,} tokens")